### Relevant Python 3.13 Changes — Advanced Tutorial Problems with Solutions

The release of Python 3.13 brought language, interpreter, typing, standard-library, diagnostics, and portability changes.

This notebook is a **second, independent advanced problem set**. It deliberately uses a tutorial style: we introduce a practical situation, inspect the old difficulty, break the task into smaller steps, and then build a complete solution.

This is still a curated selection rather than a complete list of Python 3.13 changes.

For the full release details, see the official [What's New in Python 3.13](https://docs.python.org/3.13/whatsnew/3.13.html) page.

The examples assume **Python 3.13 or newer**. Several cells deliberately run small child Python processes so that we can inspect command-line behavior and error messages without crashing the notebook kernel.

#### What this notebook practices

We will work through advanced problems involving:

- improved diagnostics,
- docstring normalization,
- class annotation scopes,
- property metadata,
- file-descriptor validation,
- archive-stream metadata,
- Unicode arrays,
- strict email parsing,
- exact rational formatting,
- glob-to-regex translation,
- MIME classification,
- fused floating-point operations,
- memory maps,
- safer `marshal` usage,
- file URI parsing,
- the new `random` CLI,
- kernel density estimation,
- SQLite lifecycle checks,
- stricter AST construction,
- and a combined asset-manifest capstone.

#### Notebook setup

Before using version-specific behavior, a production notebook should verify the interpreter instead of silently producing misleading results on an older Python.

In [1]:
import platform
import sys

print('Python:', sys.version)
print('Implementation:', platform.python_implementation())

if sys.version_info < (3, 13):
    raise RuntimeError('This notebook requires Python 3.13 or newer.')

Python: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
Implementation: CPython


We will also import a few reusable standard-library tools here. No third-party package is required by the exercises themselves.

In [2]:
from pathlib import Path, PurePosixPath, PureWindowsPath
import contextlib
import gc
import io
import math
import os
import re
import subprocess
import tempfile
import textwrap
import warnings

#### Problem 1 — Turn improved error messages into debugging evidence

Python 3.13 improves several diagnostics. Two especially practical cases are:

1. suggesting the intended keyword argument after a typo, and
2. explaining when a local script shadows a library module.

The problem is that intentionally triggering these exceptions directly in a notebook would interrupt the normal execution flow.

A safer testing technique is to run the failing expression in a **child process**, capture standard error, and then assert that the diagnostic contains the useful clue.

##### Step 1 — Capture a misspelled keyword diagnostic

In [3]:
wrong_keyword = '"alpha beta gamma".split(max_split=1)'

result = subprocess.run(
    [sys.executable, '-c', wrong_keyword],
    text=True,
    capture_output=True,
    check=False,
)

print(result.stderr)
assert result.returncode != 0
assert "Did you mean 'maxsplit'?" in result.stderr

Traceback (most recent call last):
  File "<string>", line 1, in <module>
    "alpha beta gamma".split(max_split=1)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
TypeError: split() got an unexpected keyword argument 'max_split'. Did you mean 'maxsplit'?



Notice that our test does not depend on a traceback's exact line numbers. It checks only the important semantic clue. This makes the test less brittle.

##### Step 2 — Reproduce module shadowing safely

Suppose a student creates a file named `statistics.py` and then tries to import the standard-library `statistics` module. The local file can import itself instead.

In [4]:
with tempfile.TemporaryDirectory() as tmp:
    shadow_file = Path(tmp, 'statistics.py')
    shadow_file.write_text(
        'import statistics\n'
        'print(statistics.mean([1, 2, 3]))\n',
        encoding='utf-8',
    )

    shadow_result = subprocess.run(
        [sys.executable, str(shadow_file)],
        text=True,
        capture_output=True,
        check=False,
    )

print(shadow_result.stderr)
assert shadow_result.returncode != 0
assert 'consider renaming' in shadow_result.stderr
assert 'standard library module' in shadow_result.stderr

Traceback (most recent call last):
  File "C:\Users\user1\AppData\Local\Temp\tmppck_d457\statistics.py", line 1, in <module>
    import statistics
  File "C:\Users\user1\AppData\Local\Temp\tmppck_d457\statistics.py", line 2, in <module>
    print(statistics.mean([1, 2, 3]))
          ^^^^^^^^^^^^^^^
AttributeError: module 'statistics' has no attribute 'mean' (consider renaming 'C:\Users\user1\AppData\Local\Temp\tmppck_d457\statistics.py' since it has the same name as the standard library module named 'statistics' and prevents importing that standard library module)



##### Complete solution — A reusable diagnostic runner

We can now package the child-process pattern into a helper that returns structured evidence instead of merely printing text.

In [5]:
from dataclasses import dataclass

@dataclass(frozen=True)
class DiagnosticRun:
    returncode: int
    stdout: str
    stderr: str

    @property
    def failed(self) -> bool:
        return self.returncode != 0


def run_python_snippet(source: str, *, cwd: str | Path | None = None) -> DiagnosticRun:
    completed = subprocess.run(
        [sys.executable, '-c', source],
        cwd=cwd,
        text=True,
        capture_output=True,
        check=False,
    )
    return DiagnosticRun(
        returncode=completed.returncode,
        stdout=completed.stdout,
        stderr=completed.stderr,
    )


diagnostic = run_python_snippet("'a b'.split(max_split=1)")
assert diagnostic.failed
assert 'maxsplit' in diagnostic.stderr

diagnostic

DiagnosticRun(returncode=1, stdout='', stderr='Traceback (most recent call last):\n  File \x1b"<string>"\x1b, line \x1b1\x1b, in \x1b<module>\x1b\n    \x1b\'a b\'.split\x1b\x1b(max_split=1)\x1b\n    \x1b~~~~~~~~~~~\x1b\x1b^^^^^^^^^^^^^\x1b\n\x1bTypeError\x1b: \x1bsplit() got an unexpected keyword argument \'max_split\'. Did you mean \'maxsplit\'?\x1b\n')

**Best practice:** test the presence of the meaningful recommendation, not every character of the complete traceback.

#### Problem 2 — Build executable documentation after docstring dedentation

Python 3.13 strips common leading whitespace from docstrings at compile time. This reduces bytecode-cache size and changes the raw value seen through `__doc__`.

This matters when a framework parses docstrings directly, especially when the docstring contains examples for `doctest`.

##### Step 1 — Inspect the raw docstring

In [6]:
def normalize_name(value: str) -> str:
    """
        Normalize a user-facing name.

        >>> normalize_name('  Ada   Lovelace  ')
        'Ada Lovelace'
        >>> normalize_name('')
        ''
    """
    return ' '.join(value.split())


print(repr(normalize_name.__doc__))

"\nNormalize a user-facing name.\n\n>>> normalize_name('  Ada   Lovelace  ')\n'Ada Lovelace'\n>>> normalize_name('')\n''\n"


The initial newline remains, but the common indentation that came from nesting the string inside the function body is already removed.

In [7]:
assert '\nNormalize a user-facing name.' in normalize_name.__doc__
assert '\n        Normalize a user-facing name.' not in normalize_name.__doc__

##### Step 2 — Run the examples as tests

In [8]:
import doctest

finder = doctest.DocTestFinder()
runners = []

for test in finder.find(normalize_name):
    runner = doctest.DocTestRunner(verbose=False)
    runner.run(test)
    runners.append(runner.summarize(verbose=False))

runners

[TestResults(failed=0, attempted=2)]

##### Step 3 — Extract a clean summary without double-dedenting

Older helper code sometimes applied several layers of indentation cleanup. In 3.13, a simple, explicit parser is usually easier to reason about.

In [9]:
def doc_summary(obj: object) -> str:
    doc = getattr(obj, '__doc__', None)
    if not doc:
        return ''

    for line in doc.splitlines():
        stripped = line.strip()
        if stripped:
            return stripped
    return ''


assert doc_summary(normalize_name) == 'Normalize a user-facing name.'
doc_summary(normalize_name)

'Normalize a user-facing name.'

##### Complete solution — Verify documentation and expose its summary

In [10]:
def verify_documented_callable(func):
    tests = doctest.DocTestFinder().find(func)
    runner = doctest.DocTestRunner()

    for test in tests:
        runner.run(test)

    result = runner.summarize(verbose=False)
    if result.failed:
        raise AssertionError(f'{result.failed} doctest example(s) failed')

    return {
        'name': func.__name__,
        'summary': doc_summary(func),
        'attempted_examples': result.attempted,
    }


verify_documented_callable(normalize_name)

{'name': 'normalize_name',
 'summary': 'Normalize a user-facing name.',
 'attempted_examples': 2}

#### Problem 3 — Use richer annotation scopes inside generic classes

Python 3.13 allows annotation scopes inside class scopes to contain lambdas and comprehensions.

This is an advanced feature because type aliases are lazily evaluated and the result can preserve references to the class's type parameter.

##### Step 1 — Define a lazy alias containing a lambda

In [11]:
class Envelope[T]:
    type TypeFactory = lambda: T


Envelope.TypeFactory

TypeFactory

The alias object itself is not the lambda. Its lazily evaluated value is available through `__value__`.

In [12]:
factory = Envelope.TypeFactory.__value__
print(factory)
print('Lambda returns:', factory())

<function Envelope.TypeFactory.<locals>.<lambda> at 0x0000024D0034D260>
Lambda returns: T


The lambda returns the type-parameter object `T`, not a concrete specialization such as `int`. Runtime type-alias introspection and static specialization are related but not identical concepts.

##### Step 2 — Use a comprehension in an alias

In [13]:
class Replicated[T]:
    type ThreeReferences = [T for _ in range(3)]


Replicated.ThreeReferences.__value__

[T, T, T]

##### Step 3 — Build an introspection helper

In [14]:
def describe_lazy_alias(alias) -> dict[str, object]:
    value = alias.__value__
    return {
        'alias_name': alias.__name__,
        'value_type': type(value).__name__,
        'value': value,
    }


describe_lazy_alias(Replicated.ThreeReferences)

{'alias_name': 'ThreeReferences', 'value_type': 'list', 'value': [T, T, T]}

##### Complete solution — Create generic schema metadata

In [15]:
class Schema[T]:
    type ConstructorReference = lambda: T
    type AuditColumns = [f'field_{index}' for index in range(1, 4)]


def schema_metadata(schema_class: type) -> dict[str, object]:
    constructor_ref = schema_class.ConstructorReference.__value__
    return {
        'constructor_type_parameter': constructor_ref(),
        'audit_columns': tuple(schema_class.AuditColumns.__value__),
    }


metadata = schema_metadata(Schema)
assert metadata['audit_columns'] == ('field_1', 'field_2', 'field_3')
metadata

{'constructor_type_parameter': T,
 'audit_columns': ('field_1', 'field_2', 'field_3')}

**Best practice:** keep runtime behavior out of type aliases unless laziness and annotation-scope behavior are genuinely useful. Clever aliases can become difficult to maintain.

#### Problem 4 — Use `property.__name__` in a metadata-driven API

Property objects now have a `__name__` attribute. This makes them easier to inspect without reaching into `fget`.

Suppose we are building a small serializer that exposes selected calculated fields.

##### Step 1 — Inspect property names directly

In [16]:
class Invoice:
    def __init__(self, subtotal: float, tax_rate: float):
        self.subtotal = subtotal
        self.tax_rate = tax_rate

    @property
    def tax(self) -> float:
        return self.subtotal * self.tax_rate

    @property
    def total(self) -> float:
        return self.subtotal + self.tax


print(Invoice.tax.__name__)
print(Invoice.total.__name__)

tax
total


##### Step 2 — Rename an externally exposed property

The property name is writable. A framework may use this to distinguish the Python attribute name from the external field name.

In [17]:
Invoice.tax.__name__ = 'calculated_tax'

assert Invoice.tax.__name__ == 'calculated_tax'
Invoice.tax.__name__

'calculated_tax'

##### Step 3 — Discover properties without depending on `fget`

In [18]:
def property_schema(cls: type) -> dict[str, str]:
    result = {}
    for attribute_name, value in vars(cls).items():
        if isinstance(value, property):
            result[attribute_name] = value.__name__
    return result


property_schema(Invoice)

{'tax': 'calculated_tax', 'total': 'total'}

##### Complete solution — Serialize calculated fields

In [19]:
def serialize_properties(instance: object) -> dict[str, object]:
    output = {}
    for attribute_name, descriptor in vars(type(instance)).items():
        if isinstance(descriptor, property):
            output[descriptor.__name__] = getattr(instance, attribute_name)
    return output


invoice = Invoice(125.0, 0.20)
serialized = serialize_properties(invoice)

assert serialized == {'calculated_tax': 25.0, 'total': 150.0}
serialized

{'calculated_tax': 25.0, 'total': 150.0}

#### Problem 5 — Reject booleans where a real file descriptor is required

Because `bool` is a subclass of `int`, `True` can accidentally be interpreted as file descriptor `1` and `False` as file descriptor `0`.

Python 3.13 makes a number of these mistakes easier to notice by emitting warnings.

##### Step 1 — Observe the warning without closing the underlying descriptor

In [20]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    wrapper = os.fdopen(True, mode='w', closefd=False)
    wrapper.close()

[(item.category.__name__, str(item.message)) for item in caught]

[('RuntimeWarning', 'bool is used as a file descriptor')]

We deliberately used `closefd=False`; otherwise closing the wrapper could close standard output.

In [21]:
assert any('bool is used as a file descriptor' in str(item.message) for item in caught)

##### Step 2 — Validate before calling low-level APIs

In [22]:
import operator


def require_file_descriptor(value: object) -> int:
    if isinstance(value, bool):
        raise TypeError('A boolean is not accepted as a file descriptor.')

    fd = operator.index(value)
    if fd < 0:
        raise ValueError('A file descriptor must be non-negative.')
    return fd


for invalid in (True, False, -1):
    try:
        require_file_descriptor(invalid)
    except (TypeError, ValueError) as ex:
        print(type(ex).__name__, ex)

TypeError A boolean is not accepted as a file descriptor.
TypeError A boolean is not accepted as a file descriptor.
ValueError A file descriptor must be non-negative.


##### Complete solution — A safe `fdopen` wrapper

In [23]:
def safe_fdopen(fd_like: object, mode: str = 'r', *, closefd: bool = False):
    fd = require_file_descriptor(fd_like)
    return os.fdopen(fd, mode=mode, closefd=closefd)


with tempfile.TemporaryFile(mode='w+') as temp_file:
    duplicate = os.dup(temp_file.fileno())
    try:
        with safe_fdopen(duplicate, mode='w+', closefd=True) as stream:
            stream.write('validated descriptor')
            stream.seek(0)
            assert stream.read() == 'validated descriptor'
    finally:
        # The context manager normally closes the duplicate. The guard keeps
        # cleanup correct even if the implementation changes.
        with contextlib.suppress(OSError):
            os.close(duplicate)

print('safe_fdopen completed successfully')

safe_fdopen completed successfully


#### Problem 6 — Build a uniform archive-member inspection tool

Python 3.13 adds `name` and `mode` attributes to several compressed or archived file-like objects.

This lets code inspect streams more uniformly instead of writing a different adapter for every archive module.

##### Step 1 — Inspect BZ2 and LZMA streams

In [24]:
import bz2
import lzma

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)

    with bz2.open(root / 'payload.bz2', 'wb') as bz_stream:
        bz_metadata = (str(bz_stream.name), bz_stream.mode)
        bz_stream.write(b'bz2 payload')

    with lzma.open(root / 'payload.xz', 'wb') as xz_stream:
        xz_metadata = (str(xz_stream.name), xz_stream.mode)
        xz_stream.write(b'lzma payload')

bz_metadata, xz_metadata

(('C:\\Users\\user1\\AppData\\Local\\Temp\\tmp29c73bxi\\payload.bz2', 'wb'),
 ('C:\\Users\\user1\\AppData\\Local\\Temp\\tmp29c73bxi\\payload.xz', 'wb'))

##### Step 2 — Inspect ZIP and TAR member streams

In [25]:
import tarfile
import zipfile

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)

    zip_path = root / 'bundle.zip'
    with zipfile.ZipFile(zip_path, 'w') as archive:
        archive.writestr('notes/readme.txt', 'zip content')

    with zipfile.ZipFile(zip_path) as archive:
        with archive.open('notes/readme.txt', 'r') as member:
            zip_member_metadata = (member.name, member.mode, member.read())

    tar_path = root / 'bundle.tar'
    with tarfile.open(tar_path, 'w') as archive:
        payload = b'tar content'
        info = tarfile.TarInfo('notes/readme.txt')
        info.size = len(payload)
        archive.addfile(info, io.BytesIO(payload))

    with tarfile.open(tar_path) as archive:
        member = archive.extractfile('notes/readme.txt')
        assert member is not None
        try:
            tar_member_metadata = (member.name, member.mode, member.read())
        finally:
            member.close()

zip_member_metadata, tar_member_metadata

(('notes/readme.txt', 'rb', b'zip content'),
 ('notes/readme.txt', 'rb', b'tar content'))

##### Complete solution — One metadata function for many stream types

In [26]:
def stream_identity(stream: object) -> dict[str, object]:
    return {
        'name': os.fspath(getattr(stream, 'name', '<anonymous>')),
        'mode': getattr(stream, 'mode', '<unknown>'),
        'readable': stream.readable() if hasattr(stream, 'readable') else None,
        'writable': stream.writable() if hasattr(stream, 'writable') else None,
    }


with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp, 'sample.xz')
    with lzma.open(path, 'wb') as stream:
        identity = stream_identity(stream)

identity

{'name': 'C:\\Users\\user1\\AppData\\Local\\Temp\\tmp3yt5kgt3\\sample.xz',
 'mode': 'wb',
 'readable': False,
 'writable': True}

#### Problem 7 — Store Unicode code points with `array('w')`

Python 3.13 adds the `w` array type code for Unicode characters and registers `array.array` as a `MutableSequence` by adding `clear()`.

This gives us a compact mutable sequence whose intent is clearer than the deprecated `u` code.

##### Step 1 — Round-trip non-ASCII text

In [27]:
from array import array

characters = array('w', 'Aβ🙂𐍈')
print(characters)
print(characters.tounicode())

assert characters.tounicode() == 'Aβ🙂𐍈'

array('w', 'Aβ🙂𐍈')
Aβ🙂𐍈


##### Step 2 — Treat the array as a normal mutable sequence

In [28]:
from collections.abc import MutableSequence

assert isinstance(characters, MutableSequence)

characters.append('Z')
characters[0] = 'X'

characters.tounicode()

'Xβ🙂𐍈Z'

##### Step 3 — Write a sequence-generic sanitizer

In [29]:
def remove_characters(sequence: MutableSequence[str], forbidden: set[str]) -> None:
    index = 0
    while index < len(sequence):
        if sequence[index] in forbidden:
            del sequence[index]
        else:
            index += 1


remove_characters(characters, {'β', 'Z'})
assert characters.tounicode() == 'X🙂𐍈'
characters

array('w', 'X🙂𐍈')

##### Complete solution — Reuse storage safely

In [30]:
def refill_unicode_buffer(buffer: array, text: str) -> array:
    if buffer.typecode != 'w':
        raise TypeError("Expected an array with type code 'w'.")
    buffer.clear()
    buffer.extend(text)
    return buffer


refill_unicode_buffer(characters, 'Python 🐍 3.13')
assert characters.tounicode() == 'Python 🐍 3.13'
characters

array('w', 'Python 🐍 3.13')

#### Problem 8 — Reject ambiguous email addresses with strict parsing

In Python 3.13, `email.utils.parseaddr()` and `getaddresses()` use stricter parsing by default.

The goal is not to prove that an address exists. The goal is to avoid quietly converting malformed input into a misleading address.

##### Step 1 — Compare strict and compatibility behavior

In [31]:
from email.utils import parseaddr

malformed = 'alice@example.com <bob@example.com>'

print('strict=True :', parseaddr(malformed, strict=True))
print('strict=False:', parseaddr(malformed, strict=False))

strict=True : ('', '')
strict=False: ('', 'alice@example.com')


Strict mode returns an empty pair for this ambiguous input rather than pretending one interpretation is correct.

##### Step 2 — Handle a header-injection-shaped string

In [32]:
header_like = 'Bad <bad@example.com\nBcc: victim@example.com>'

strict_result = parseaddr(header_like, strict=True)
compat_result = parseaddr(header_like, strict=False)

strict_result, compat_result

(('', ''), ('Bad', 'bad@example.com'))

##### Step 3 — Build a boundary validator

In [33]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Mailbox:
    display_name: str
    address: str


def parse_mailbox(value: str) -> Mailbox:
    display_name, address = parseaddr(value, strict=True)

    if not address:
        raise ValueError(f'Invalid or ambiguous mailbox: {value!r}')
    if '\n' in value or '\r' in value:
        raise ValueError('Newlines are not permitted in a mailbox field.')
    if '@' not in address:
        raise ValueError(f'Missing @ in mailbox address: {address!r}')

    return Mailbox(display_name=display_name, address=address)


parse_mailbox('Ada Lovelace <ada@example.org>')

Mailbox(display_name='Ada Lovelace', address='ada@example.org')

##### Complete solution — Validate a recipient batch without partial acceptance

In [34]:
def parse_recipient_batch(values: list[str]) -> tuple[Mailbox, ...]:
    parsed = []
    errors = []

    for index, value in enumerate(values):
        try:
            parsed.append(parse_mailbox(value))
        except ValueError as ex:
            errors.append(f'item {index}: {ex}')

    if errors:
        raise ValueError('Recipient batch rejected:\n' + '\n'.join(errors))

    return tuple(parsed)


valid_batch = parse_recipient_batch([
    'Ada <ada@example.org>',
    'grace@example.org',
])

assert len(valid_batch) == 2
valid_batch

(Mailbox(display_name='Ada', address='ada@example.org'),
 Mailbox(display_name='', address='grace@example.org'))

#### Problem 9 — Format exact rational values as polished reports

`fractions.Fraction` now supports the standard formatting rules for fill, alignment, sign, minimum width, and grouping.

This is useful when we need exact arithmetic but still want table-like output.

##### Step 1 — Keep the calculation exact

In [35]:
from fractions import Fraction

conversion_rate = Fraction(1234567, 1000)
commission_rate = Fraction(7, 200)

commission = conversion_rate * commission_rate

conversion_rate, commission

(Fraction(1234567, 1000), Fraction(8641969, 200000))

##### Step 2 — Use numeric presentation types

In [36]:
print(f'Rate       : {conversion_rate:>18,.2f}')
print(f'Commission : {commission:>18,.4f}')

Rate       :           1,234.57
Commission :            43.2098


##### Step 3 — Use fill, sign, width, and grouping together

In [37]:
formatted = format(conversion_rate, '+020,.2f')
print(formatted)
assert formatted.startswith('+')
assert ',' in formatted

+0,000,000,001,234.57


##### Complete solution — Render a reusable exact report

In [38]:
def exact_report(rows: list[tuple[str, Fraction]], *, width: int = 18) -> str:
    lines = []
    for label, value in rows:
        lines.append(f'{label:<16} {value:>{width},.4f}')
    return '\n'.join(lines)


report = exact_report([
    ('conversion rate', conversion_rate),
    ('commission rate', commission_rate),
    ('commission', commission),
])

print(report)

conversion rate          1,234.5670
commission rate              0.0350
commission                  43.2098


The displayed decimal is rounded for presentation, while the underlying values remain exact fractions.

#### Problem 10 — Compile glob patterns and match virtual paths

Python 3.13 adds `glob.translate()` and `PurePath.full_match()`.

These solve related but different problems:

- `glob.translate()` converts a shell-style path pattern into a regular-expression string.
- `PurePath.full_match()` checks whether one path matches the complete pattern, including recursive `**` segments.

##### Step 1 — Translate a recursive glob

In [39]:
import glob

pattern = 'src/**/*.py'
regex_text = glob.translate(pattern, recursive=True)
regex = re.compile(regex_text)

print(regex_text)

(?s:src[\\/](?:[^\\/.][^\\/]*[\\/])*(?!\.)[^\\/]*\.py)\Z


##### Step 2 — Filter strings without touching the filesystem

In [40]:
candidates = [
    'src/main.py',
    'src/package/api.py',
    'src/package/data.json',
    'tests/test_api.py',
]

regex_matches = [path for path in candidates if regex.fullmatch(path)]
regex_matches

['src/main.py', 'src/package/api.py']

##### Step 3 — Match with `PurePath.full_match()`

In [41]:
path_matches = [
    path
    for path in candidates
    if PurePosixPath(path).full_match(pattern)
]

assert path_matches == regex_matches
path_matches

['src/main.py', 'src/package/api.py']

##### Step 4 — Make case sensitivity explicit

In [42]:
windows_style = PureWindowsPath('SRC/Package/API.PY')

print(windows_style.full_match('src/**/*.py'))
print(windows_style.full_match('src/**/*.py', case_sensitive=True))
print(windows_style.full_match('src/**/*.py', case_sensitive=False))

True
False
True


##### Complete solution — A virtual manifest filter

In [43]:
def select_virtual_paths(
    paths: list[str],
    pattern: str,
    *,
    windows_rules: bool = False,
    case_sensitive: bool | None = None,
) -> list[str]:
    path_type = PureWindowsPath if windows_rules else PurePosixPath
    return [
        value
        for value in paths
        if path_type(value).full_match(pattern, case_sensitive=case_sensitive)
    ]


selected = select_virtual_paths(
    ['Assets/Logo.PNG', 'assets/icon.png', 'assets/docs/readme.md'],
    'assets/**/*.png',
    windows_rules=True,
    case_sensitive=False,
)

selected

['Assets/Logo.PNG', 'assets/icon.png']

#### Problem 11 — Classify filesystem paths with `mimetypes.guess_file_type()`

`mimetypes.guess_type()` historically accepted both URLs and file paths. Python 3.13 adds `guess_file_type()` so code can communicate that it is classifying a filesystem path.

##### Step 1 — Classify common files

In [44]:
import mimetypes

for sample in [
    Path('report.csv'),
    Path('image.png'),
    Path('backup.tar.gz'),
    Path('README'),
]:
    print(sample, '->', mimetypes.guess_file_type(sample))

report.csv -> ('application/vnd.ms-excel', None)
image.png -> ('image/png', None)
backup.tar.gz -> ('application/x-tar', 'gzip')
README -> (None, None)


The result is a `(MIME type, content encoding)` pair. A compressed archive can have both values.

##### Step 2 — Turn the pair into a policy decision

In [45]:
@dataclass(frozen=True)
class FileClassification:
    path: Path
    media_type: str | None
    encoding: str | None

    @property
    def is_image(self) -> bool:
        return bool(self.media_type and self.media_type.startswith('image/'))


def classify_file(path: str | os.PathLike[str]) -> FileClassification:
    normalized = Path(path)
    media_type, encoding = mimetypes.guess_file_type(normalized)
    return FileClassification(normalized, media_type, encoding)


classify_file('diagram.svg')

FileClassification(path=WindowsPath('diagram.svg'), media_type='image/svg+xml', encoding=None)

##### Complete solution — Enforce a portable upload allowlist

MIME databases are platform-dependent. For example, some systems classify `.csv` as `text/csv`, while others use the historical alias `application/vnd.ms-excel`. Therefore, production code should not assert one universal MIME string for a filename.

The policy below checks all of the following:

1. The filename suffix is explicitly allowed.
2. A transport encoding such as `gzip` is rejected.
3. The guessed MIME type is one of the known aliases for that suffix.

> **Security note:** this is filename classification, not content verification. A real upload service should inspect file signatures or parse the content before trusting it.


In [46]:
ALLOWED_UPLOAD_TYPES: dict[str, frozenset[str]] = {
    '.csv': frozenset({
        'text/csv',
        'application/csv',
        'text/x-csv',
        'application/vnd.ms-excel',
    }),
    '.json': frozenset({'application/json'}),
    '.png': frozenset({'image/png'}),
    '.jpg': frozenset({'image/jpeg'}),
    '.jpeg': frozenset({'image/jpeg'}),
}


def validate_upload_name(path: str | os.PathLike[str]) -> FileClassification:
    classification = classify_file(path)
    suffix = classification.path.suffix.lower()

    accepted_media_types = ALLOWED_UPLOAD_TYPES.get(suffix)
    if accepted_media_types is None:
        display_suffix = suffix or '<none>'
        raise ValueError(f'Unsupported filename suffix: {display_suffix!r}')

    if classification.encoding is not None:
        raise ValueError(
            f'Encoded uploads are not accepted: {classification.encoding}'
        )

    if classification.media_type not in accepted_media_types:
        raise ValueError(
            'MIME type does not match the allowed aliases for '
            f'{suffix}: {classification.media_type!r}'
        )

    return classification


csv_upload = validate_upload_name('customers.csv')
assert csv_upload.path.suffix.lower() == '.csv'
assert csv_upload.media_type in ALLOWED_UPLOAD_TYPES['.csv']
print('CSV classification:', csv_upload)

try:
    validate_upload_name('backup.tar.gz')
except ValueError as ex:
    print('Rejected archive:', ex)


CSV classification: FileClassification(path=WindowsPath('customers.csv'), media_type='application/vnd.ms-excel', encoding=None)
Rejected archive: Unsupported filename suffix: '.gz'


#### Problem 12 — Reduce floating-point rounding with `math.fma()`

`math.fma(x, y, z)` computes `x * y + z` with a single rounding step.

The ordinary expression first rounds the multiplication and then rounds the addition. When large nearly-cancelling values are involved, the difference can be visible.

##### Step 1 — Find a cancellation example

In [47]:
x = 1e16
y = 1.0000000000000002
z = -1e16

ordinary = x * y + z
fused = math.fma(x, y, z)

ordinary, fused, fused - ordinary

(2.0, 2.220446049250313, 0.22044604925031308)

Neither result should be confused with arbitrary-precision arithmetic. `fma` merely avoids the intermediate rounding of the product.

##### Step 2 — Use FMA in Horner's method

In [48]:
def evaluate_polynomial_fma(coefficients: list[float], x: float) -> float:
    if not coefficients:
        return 0.0

    result = coefficients[0]
    for coefficient in coefficients[1:]:
        result = math.fma(result, x, coefficient)
    return result


def evaluate_polynomial_plain(coefficients: list[float], x: float) -> float:
    if not coefficients:
        return 0.0

    result = coefficients[0]
    for coefficient in coefficients[1:]:
        result = result * x + coefficient
    return result

##### Step 3 — Compare against a high-precision reference

In [49]:
from decimal import Decimal, localcontext

coefficients = [1e16, -1e16, 3.0, -2.0]
point = 1.0000000000000002

with localcontext() as ctx:
    ctx.prec = 60
    dx = Decimal.from_float(point)
    reference = Decimal.from_float(coefficients[0])
    for coefficient in coefficients[1:]:
        reference = reference * dx + Decimal.from_float(coefficient)

plain_value = evaluate_polynomial_plain(coefficients, point)
fma_value = evaluate_polynomial_fma(coefficients, point)

plain_error = abs(Decimal.from_float(plain_value) - reference)
fma_error = abs(Decimal.from_float(fma_value) - reference)

print('plain:', plain_value, 'error:', plain_error)
print('fma  :', fma_value, 'error:', fma_error)

plain: 3.000000000000001 error: 0.2204460492503138448787899374
fma  : 3.2204460492503153 error: 5.682361029489542903519811033E-16


##### Complete solution — Select the numerically better implementation

In [50]:
assert fma_error <= plain_error

result_summary = {
    'plain': plain_value,
    'fma': fma_value,
    'plain_error': plain_error,
    'fma_error': fma_error,
}

result_summary

{'plain': 3.000000000000001,
 'fma': 3.2204460492503153,
 'plain_error': Decimal('0.2204460492503138448787899374'),
 'fma_error': Decimal('5.682361029489542903519811033E-16')}

#### Problem 13 — Treat `mmap` as an explicitly seekable binary stream

Python 3.13 adds `mmap.seekable()` and makes `mmap.seek()` return the new absolute position.

This improves compatibility with APIs that expect a seekable file-like object.

##### Step 1 — Create a small binary record file

In [51]:
import mmap

records = b'HEAD' + b'0005' + b'hello' + b'TAIL'

with tempfile.NamedTemporaryFile(delete=False) as temp:
    binary_path = Path(temp.name)
    temp.write(records)

binary_path

WindowsPath('C:/Users/user1/AppData/Local/Temp/tmp5emfpxck')

##### Step 2 — Inspect stream capabilities and positions

In [52]:
try:
    with binary_path.open('r+b') as file_obj:
        with mmap.mmap(file_obj.fileno(), 0) as mapped:
            assert mapped.seekable() is True
            new_position = mapped.seek(4)
            length = int(mapped.read(4))
            payload = mapped.read(length)

    print('new position:', new_position)
    print('length:', length)
    print('payload:', payload)
finally:
    binary_path.unlink(missing_ok=True)

new position: 4
length: 5
payload: b'hello'


##### Complete solution — Parse a framed record from any seekable binary object

In [53]:
def parse_framed_record(stream) -> bytes:
    if not stream.seekable():
        raise ValueError('The input stream must be seekable.')

    stream.seek(0)
    if stream.read(4) != b'HEAD':
        raise ValueError('Missing HEAD marker.')

    size_bytes = stream.read(4)
    if len(size_bytes) != 4 or not size_bytes.isdigit():
        raise ValueError('Invalid four-byte decimal size.')

    expected_size = int(size_bytes)
    payload = stream.read(expected_size)
    if len(payload) != expected_size:
        raise ValueError('Truncated payload.')

    if stream.read(4) != b'TAIL':
        raise ValueError('Missing TAIL marker.')

    return payload


with tempfile.NamedTemporaryFile() as temp:
    temp.write(records)
    temp.flush()
    with mmap.mmap(temp.fileno(), 0, access=mmap.ACCESS_READ) as mapped:
        assert parse_framed_record(mapped) == b'hello'

print('framed record parsed')

framed record parsed


#### Problem 14 — Reject executable code objects in `marshal` data

Python 3.13 adds an `allow_code` parameter to `marshal` operations.

`marshal` is not a secure format for untrusted data. The new flag does **not** turn it into one. It does, however, let an application reject code objects when its format is supposed to contain data only.

##### Step 1 — Serialize ordinary data with code disabled

In [54]:
import marshal

cache_record = {
    'version': 3,
    'items': ['alpha', 'beta'],
    'enabled': True,
}

payload = marshal.dumps(cache_record, allow_code=False)
restored = marshal.loads(payload, allow_code=False)

assert restored == cache_record
restored

{'version': 3, 'items': ['alpha', 'beta'], 'enabled': True}

##### Step 2 — Reject a code object during serialization

In [55]:
compiled = compile('40 + 2', '<cache-entry>', 'eval')

try:
    marshal.dumps(compiled, allow_code=False)
except ValueError as ex:
    print(type(ex).__name__, ex)

ValueError marshalling code objects is disallowed


##### Step 3 — Reject a code object during deserialization

In [56]:
legacy_payload = marshal.dumps(compiled)

try:
    marshal.loads(legacy_payload, allow_code=False)
except ValueError as ex:
    print(type(ex).__name__, ex)

ValueError unmarshalling code objects is disallowed


##### Complete solution — A versioned data-only cache

In [57]:
CACHE_VERSION = 1


def dump_data_cache(data: object) -> bytes:
    envelope = {'cache_version': CACHE_VERSION, 'data': data}
    return marshal.dumps(envelope, allow_code=False)


def load_data_cache(payload: bytes) -> object:
    envelope = marshal.loads(payload, allow_code=False)

    if not isinstance(envelope, dict):
        raise ValueError('Cache envelope must be a dictionary.')
    if envelope.get('cache_version') != CACHE_VERSION:
        raise ValueError('Unsupported cache version.')
    if 'data' not in envelope:
        raise ValueError('Cache data is missing.')

    return envelope['data']


encoded_cache = dump_data_cache([1, 2, 3])
assert load_data_cache(encoded_cache) == [1, 2, 3]
len(encoded_cache)

48

#### Problem 15 — Round-trip absolute paths through file URIs

Python 3.13 adds `Path.from_uri()` and exposes the low-level parser used by pure paths through `PurePath.parser`.

This is useful when a configuration or manifest stores local files as `file:` URIs.

##### Step 1 — Convert an absolute path to a URI and back

In [58]:
with tempfile.TemporaryDirectory() as tmp:
    original_path = Path(tmp, 'data file.txt').resolve()
    original_path.write_text('content', encoding='utf-8')

    file_uri = original_path.as_uri()
    restored_path = Path.from_uri(file_uri)

    print(file_uri)
    print(restored_path)
    assert restored_path == original_path

file:///C:/Users/user1/AppData/Local/Temp/tmpttehfj4e/data%20file.txt
C:\Users\user1\AppData\Local\Temp\tmpttehfj4e\data file.txt


##### Step 2 — Reject non-file schemes

In [59]:
for invalid_uri in [
    'https://example.org/file.txt',
    'file:relative/path.txt',
]:
    try:
        Path.from_uri(invalid_uri)
    except ValueError as ex:
        print(invalid_uri, '->', ex)

https://example.org/file.txt -> URI does not start with 'file:': 'https://example.org/file.txt'
file:relative/path.txt -> URI is not absolute: 'file:relative/path.txt'


##### Step 3 — Inspect platform-specific parsers

In [60]:
print('POSIX parser  :', PurePosixPath.parser.__name__)
print('Windows parser:', PureWindowsPath.parser.__name__)

POSIX parser  : posixpath
Windows parser: ntpath


##### Step 4 — Catch the more specific unsupported-operation exception

In [61]:
from pathlib import PosixPath, WindowsPath, UnsupportedOperation

try:
    incompatible = WindowsPath('C:/temporary') if os.name != 'nt' else PosixPath('/temporary')
except UnsupportedOperation as ex:
    print(type(ex).__name__, ex)

UnsupportedOperation cannot instantiate 'PosixPath' on your system


##### Complete solution — Normalize a manifest path reference

In [62]:
def path_from_reference(reference: str | os.PathLike[str]) -> Path:
    if isinstance(reference, str) and reference.startswith('file:'):
        return Path.from_uri(reference)

    return Path(reference).expanduser().absolute()


with tempfile.TemporaryDirectory() as tmp:
    sample = Path(tmp, 'sample.txt').resolve()
    by_uri = path_from_reference(sample.as_uri())
    by_path = path_from_reference(sample)

assert by_uri == by_path == sample
sample

WindowsPath('C:/Users/user1/AppData/Local/Temp/tmpk0b_db_g/sample.txt')

#### Problem 16 — Use the `random` module from the command line

Python 3.13 adds a command-line interface to the `random` module.

The output is intentionally non-deterministic, so a good automated test validates the **contract** rather than a specific random value.

##### Step 1 — Generate a bounded integer

In [63]:
integer_run = subprocess.run(
    [sys.executable, '-m', 'random', '--integer', '6'],
    text=True,
    capture_output=True,
    check=True,
)

rolled = int(integer_run.stdout.strip())
print('rolled:', rolled)
assert 1 <= rolled <= 6

rolled: 6


##### Step 2 — Choose from explicit values

In [64]:
options = ['red', 'green', 'blue']
choice_run = subprocess.run(
    [sys.executable, '-m', 'random', '--choice', *options],
    text=True,
    capture_output=True,
    check=True,
)

chosen = choice_run.stdout.strip()
print('chosen:', chosen)
assert chosen in options

chosen: red


##### Step 3 — Generate a bounded float

In [65]:
float_run = subprocess.run(
    [sys.executable, '-m', 'random', '--float', '2.5'],
    text=True,
    capture_output=True,
    check=True,
)

sampled = float(float_run.stdout.strip())
print('sampled:', sampled)
assert 0.0 <= sampled <= 2.5

sampled: 2.2647348362743385


##### Complete solution — A validated CLI wrapper

In [66]:
def random_cli(*arguments: str) -> str:
    completed = subprocess.run(
        [sys.executable, '-m', 'random', *arguments],
        text=True,
        capture_output=True,
        check=False,
    )

    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or 'random CLI failed')

    return completed.stdout.strip()


value = int(random_cli('--integer', '10'))
assert 1 <= value <= 10
value

2

#### Problem 17 — Estimate and sample a continuous distribution with `statistics.kde()`

Python 3.13 adds kernel density estimation through `statistics.kde()` and sampling through `statistics.kde_random()`.

A KDE turns a finite sample into a smooth estimated probability-density function. The bandwidth controls smoothing and is a modeling choice, not a universal constant.

##### Step 1 — Create a small observed sample

In [67]:
from statistics import kde, kde_random

observations = [
    1.0, 1.1, 1.2, 1.25, 1.4,
    2.0, 2.1, 2.15, 2.2, 2.4,
]

bandwidth = 0.25
density = kde(observations, h=bandwidth, kernel='normal')

##### Step 2 — Evaluate the density at selected points

In [68]:
for point in [0.5, 1.2, 1.7, 2.2, 3.0]:
    print(f'{point:>4.1f} -> {density(point):.6f}')

 0.5 -> 0.035738
 1.2 -> 0.696426
 1.7 -> 0.321360
 2.2 -> 0.696191
 3.0 -> 0.010703


##### Step 3 — Check that the numerical area is approximately one

In [69]:
left = -1.0
right = 5.0
steps = 20_000
width = (right - left) / steps

area = sum(
    density(left + (index + 0.5) * width) * width
    for index in range(steps)
)

print('approximate area:', area)
assert 0.995 < area < 1.005

approximate area: 1.0


##### Step 4 — Produce reproducible simulated values

In [70]:
sampler = kde_random(observations, h=bandwidth, kernel='normal', seed=2026)
simulated = [sampler() for _ in range(8)]

simulated

[0.9827147997520441,
 2.2944417648458493,
 1.3989590994040682,
 2.477345227534095,
 2.295909208995103,
 2.4510944569151514,
 2.319430457032361,
 2.3629516032167093]

##### Complete solution — Summarize density regions

In [71]:
def densest_grid_points(
    data: list[float],
    *,
    bandwidth: float,
    grid_start: float,
    grid_stop: float,
    grid_size: int = 101,
    top_n: int = 5,
) -> list[tuple[float, float]]:
    estimate = kde(data, h=bandwidth, kernel='normal')
    spacing = (grid_stop - grid_start) / (grid_size - 1)
    scored = [
        (grid_start + index * spacing, estimate(grid_start + index * spacing))
        for index in range(grid_size)
    ]
    return sorted(scored, key=lambda item: item[1], reverse=True)[:top_n]


densest_grid_points(
    observations,
    bandwidth=bandwidth,
    grid_start=0.0,
    grid_stop=3.0,
)

[(2.16, 0.7046322402123004),
 (2.13, 0.7021683805024785),
 (2.19, 0.6995427819535742),
 (1.2, 0.6964255795141614),
 (1.17, 0.6955484040842568)]

#### Problem 18 — Make SQLite resource ownership and selective dumps explicit

Python 3.13 emits a `ResourceWarning` when a `sqlite3.Connection` is garbage-collected without being explicitly closed.

It also adds a `filter` parameter to `Connection.iterdump()`, which can restrict a dump to matching database objects.

##### Step 1 — Demonstrate the lifecycle warning

In [72]:
import sqlite3

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', ResourceWarning)
    leaked_connection = sqlite3.connect(':memory:')
    leaked_connection.execute('CREATE TABLE sample(value)')
    del leaked_connection
    gc.collect()

resource_messages = [
    str(item.message)
    for item in caught
    if issubclass(item.category, ResourceWarning)
]

resource_messages

['unclosed database in <sqlite3.Connection object at 0x0000024D0039C310>']

In [73]:
assert any('unclosed database' in message for message in resource_messages)

##### Step 2 — Use explicit close ownership

A connection used as a context manager commits or rolls back transactions, but the connection itself should still be closed explicitly. `contextlib.closing()` makes both responsibilities visible.

In [74]:
with contextlib.closing(sqlite3.connect(':memory:')) as connection:
    with connection:
        connection.execute('CREATE TABLE users(id INTEGER, name TEXT)')
        connection.execute('CREATE TABLE audit(id INTEGER, action TEXT)')
        connection.execute('INSERT INTO users VALUES (?, ?)', (1, 'Ada'))
        connection.execute('INSERT INTO audit VALUES (?, ?)', (1, 'created'))

    user_dump = '\n'.join(connection.iterdump(filter='user%'))

print(user_dump)
assert 'CREATE TABLE users' in user_dump
assert 'CREATE TABLE audit' not in user_dump

BEGIN TRANSACTION;
CREATE TABLE users(id INTEGER, name TEXT);
INSERT INTO "users" VALUES(1,'Ada');
COMMIT;


##### Complete solution — Dump one logical subsystem

In [75]:
def dump_matching_objects(connection: sqlite3.Connection, pattern: str) -> str:
    statements = list(connection.iterdump(filter=pattern))
    if statements == ['BEGIN TRANSACTION;', 'COMMIT;']:
        raise LookupError(f'No database objects matched {pattern!r}.')
    return '\n'.join(statements)


with contextlib.closing(sqlite3.connect(':memory:')) as connection:
    connection.executescript("""
        CREATE TABLE billing_invoice(id INTEGER, amount REAL);
        CREATE TABLE billing_payment(id INTEGER, amount REAL);
        CREATE TABLE account_user(id INTEGER, name TEXT);
        INSERT INTO billing_invoice VALUES (1, 125.0);
    """)
    billing_dump = dump_matching_objects(connection, 'billing_%')

print(billing_dump)

BEGIN TRANSACTION;
CREATE TABLE billing_invoice(id INTEGER, amount REAL);
INSERT INTO "billing_invoice" VALUES(1,125.0);
CREATE TABLE billing_payment(id INTEGER, amount REAL);
COMMIT;


#### Problem 19 — Construct AST nodes in a forward-compatible way

Python 3.13 makes AST node constructors stricter and gives omitted optional, list, and expression-context fields more predictable defaults.

Omitting required fields or inventing unknown keyword fields now raises a `DeprecationWarning`; these cases are scheduled to become errors in Python 3.15.

##### Step 1 — Observe useful defaults

In [76]:
import ast

name_node = ast.Name(id='customer')
call_node = ast.Call(func=name_node)

print(ast.dump(name_node, include_attributes=False))
print(ast.dump(call_node, include_attributes=False))

assert isinstance(name_node.ctx, ast.Load)
assert call_node.args == []
assert call_node.keywords == []

Name(id='customer', ctx=Load())
Call(func=Name(id='customer', ctx=Load()))


##### Step 2 — Detect a missing required field

In [77]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', DeprecationWarning)
    incomplete = ast.BinOp(left=ast.Constant(1), right=ast.Constant(2))

[(item.category.__name__, str(item.message)) for item in caught]

[('DeprecationWarning',
  "BinOp.__init__ missing 1 required positional argument: 'op'. This will become an error in Python 3.15.")]

The `op` field is required. The object can currently be created with a warning, but code should treat that warning as a migration failure.

##### Step 3 — Detect an unknown field

In [78]:
with warnings.catch_warnings(record=True) as caught_unknown:
    warnings.simplefilter('always', DeprecationWarning)
    unusual = ast.Constant(value=42, made_up_metadata='not an AST field')

[(item.category.__name__, str(item.message)) for item in caught_unknown]

[('DeprecationWarning',
  "Constant.__init__ got an unexpected keyword argument 'made_up_metadata'. Support for arbitrary keyword arguments is deprecated and will be removed in Python 3.15.")]

##### Complete solution — Build and validate an expression tree

In [79]:
def build_weighted_total_expression() -> ast.Expression:
    tree = ast.Expression(
        body=ast.BinOp(
            left=ast.BinOp(
                left=ast.Name(id='price'),
                op=ast.Mult(),
                right=ast.Name(id='quantity'),
            ),
            op=ast.Add(),
            right=ast.Name(id='shipping'),
        )
    )
    ast.fix_missing_locations(tree)
    return tree


expression_tree = build_weighted_total_expression()
compiled_expression = compile(expression_tree, '<generated>', 'eval')
value = eval(
    compiled_expression,
    {'__builtins__': {}},
    {'price': 12.5, 'quantity': 4, 'shipping': 5.0},
)

assert value == 55.0
ast.dump(expression_tree, indent=2)

"Expression(\n  body=BinOp(\n    left=BinOp(\n      left=Name(id='price', ctx=Load()),\n      op=Mult(),\n      right=Name(id='quantity', ctx=Load())),\n    op=Add(),\n    right=Name(id='shipping', ctx=Load())))"

**Best practice:** always provide every required AST field explicitly and run tests with deprecation warnings enabled.

#### Problem 20 — Capstone: validate an asset manifest step by step

We will now combine several independent Python 3.13 changes into one realistic boundary-validation problem.

Each manifest entry contains:

- a local file reference, which may be a path or `file:` URI,
- an owner mailbox,
- an expected glob pattern,
- and a logical size represented exactly as a fraction.

Our validator should:

1. normalize the path reference,
2. require a full glob match,
3. classify the filename with `guess_file_type()`,
4. parse the mailbox strictly,
5. and format an exact size report.

##### Step 1 — Define the input and output records

In [80]:
@dataclass(frozen=True)
class AssetRequest:
    reference: str
    owner: str
    pattern: str
    logical_kib: Fraction


@dataclass(frozen=True)
class ValidatedAsset:
    path: Path
    owner: Mailbox
    media_type: str
    logical_kib: Fraction

##### Step 2 — Validate one entry

In [81]:
def validate_asset(request: AssetRequest) -> ValidatedAsset:
    path = path_from_reference(request.reference)

    # Convert separators to POSIX form so one glob syntax works on every OS.
    # The pattern itself must remain root-agnostic (for example, ``**/assets/*.png``)
    # because absolute Windows paths begin with a drive, not ``/``.
    match_candidate = PurePosixPath(path.as_posix())
    if not match_candidate.full_match(request.pattern):
        raise ValueError(
            f'{match_candidate.as_posix()!r} does not match '
            f'portable pattern {request.pattern!r}.'
        )

    classification = classify_file(path)
    if classification.media_type is None:
        raise ValueError(f'Cannot determine media type for {path.name!r}.')
    if classification.encoding is not None:
        raise ValueError(f'Encoded asset is not accepted: {classification.encoding}.')

    owner = parse_mailbox(request.owner)

    if request.logical_kib < 0:
        raise ValueError('Logical size cannot be negative.')

    return ValidatedAsset(
        path=path,
        owner=owner,
        media_type=classification.media_type,
        logical_kib=request.logical_kib,
    )


##### Step 3 — Create temporary assets and validate them

The glob patterns deliberately start with `**`, not `/**`. The latter is anchored to a POSIX root and fails for absolute Windows paths such as `C:/Users/...`. A root-agnostic pattern still performs a full match while allowing the platform-specific prefix to be consumed by `**`.

In [82]:
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp).resolve()
    image_path = root / 'assets' / 'images' / 'logo.png'
    data_path = root / 'assets' / 'data' / 'customers.csv'

    image_path.parent.mkdir(parents=True)
    data_path.parent.mkdir(parents=True)
    image_path.write_bytes(b'not a real PNG; filename classification only')
    data_path.write_text('id,name\n1,Ada\n', encoding='utf-8')

    requests = [
        AssetRequest(
            reference=image_path.as_uri(),
            owner='Designer <designer@example.org>',
            # No leading slash: works for both /tmp/... and C:/Users/... paths.
            pattern='**/assets/images/*.png',
            logical_kib=Fraction(1537, 10),
        ),
        AssetRequest(
            reference=str(data_path),
            owner='Data Team <data@example.org>',
            pattern='**/assets/data/*.csv',
            logical_kib=Fraction(2049, 20),
        ),
    ]

    validated_assets = [validate_asset(item) for item in requests]

validated_assets


[ValidatedAsset(path=WindowsPath('C:/Users/user1/AppData/Local/Temp/tmpz5afn7hg/assets/images/logo.png'), owner=Mailbox(display_name='Designer', address='designer@example.org'), media_type='image/png', logical_kib=Fraction(1537, 10)),
 ValidatedAsset(path=WindowsPath('C:/Users/user1/AppData/Local/Temp/tmpz5afn7hg/assets/data/customers.csv'), owner=Mailbox(display_name='Data Team', address='data@example.org'), media_type='application/vnd.ms-excel', logical_kib=Fraction(2049, 20))]

##### Step 4 — Produce a deterministic report

In [83]:
def asset_report(assets: list[ValidatedAsset]) -> str:
    header = f"{'FILE':<24} {'OWNER':<24} {'MIME TYPE':<22} {'KiB':>12}"
    rule = '-' * len(header)
    rows = [header, rule]

    for asset in assets:
        rows.append(
            f'{asset.path.name:<24} '
            f'{asset.owner.address:<24} '
            f'{asset.media_type:<22} '
            f'{asset.logical_kib:>12,.2f}'
        )

    total = sum((asset.logical_kib for asset in assets), start=Fraction(0))
    rows.extend([rule, f"{'TOTAL':<72} {total:>12,.2f}"])
    return '\n'.join(rows)


print(asset_report(validated_assets))

FILE                     OWNER                    MIME TYPE                       KiB
-------------------------------------------------------------------------------------
logo.png                 designer@example.org     image/png                    153.70
customers.csv            data@example.org         application/vnd.ms-excel       102.45
-------------------------------------------------------------------------------------
TOTAL                                                                          256.15


##### Step 5 — Verify failure paths

In [84]:
bad_requests = [
    AssetRequest(
        reference='https://example.org/logo.png',
        owner='owner@example.org',
        pattern='**/*.png',
        logical_kib=Fraction(1),
    ),
    AssetRequest(
        reference='/tmp/archive.tar.gz',
        owner='owner@example.org',
        pattern='**/*.tar.gz',
        logical_kib=Fraction(1),
    ),
    AssetRequest(
        reference='/tmp/logo.png',
        owner='alice@example.org <bob@example.org>',
        pattern='**/*.png',
        logical_kib=Fraction(1),
    ),
]

for request in bad_requests:
    try:
        validate_asset(request)
    except (ValueError, OSError) as ex:
        print(type(ex).__name__, ex)

ValueError Encoded asset is not accepted: gzip.
ValueError Invalid or ambiguous mailbox: 'alice@example.org <bob@example.org>'


The capstone uses Python 3.13 additions to improve **boundaries** rather than merely shorten syntax:

- explicit file-URI parsing,
- full-path glob semantics,
- filesystem-focused MIME guessing,
- strict mailbox parsing,
- and richer exact-number formatting.

#### Final review

A strong migration to Python 3.13 is not just a search-and-replace exercise.

The most valuable changes often help code state its assumptions more clearly:

- diagnostics can become regression tests,
- warnings can reveal ambiguous API use,
- new metadata can remove private introspection,
- stricter parsers can protect application boundaries,
- numerical primitives can improve precision,
- resource warnings can expose lifecycle bugs,
- and deprecations can be treated as future failures during CI.

#### Suggested extension exercises

1. Extend the asset capstone so that archive members are inspected with the uniform `name` and `mode` interface.
2. Add a KDE-based anomaly score for asset sizes.
3. Generate the validation expression with AST nodes and reject every `DeprecationWarning` as an error.
4. Add a subprocess test proving that a deliberately shadowed module name produces the expected Python 3.13 recommendation.
5. Build a small command-line tool that uses the `random` CLI only for demonstration mode, while using a seeded `random.Random` instance in tests.